# Topic 03: Time Series Mastery

This notebook covers high-performance time series operations. In a FAB environment, sensors produce data continuously, so mastering time-based manipulation is critical for analysis.

In [1]:
import pandas as pd
import os

DATA_PATH = "../data/sensors_sample.csv"
df = pd.read_csv(DATA_PATH, index_col=0)
df['ts'] = pd.to_datetime(df['ts'])
df = df.set_index('ts').sort_index()

## 1. Precise Time Slicing
Selecting data within a specific time range using `pd.Timedelta`.

In [2]:
start_time = df.index[0]
end_time = start_time + pd.Timedelta(hours=4)

print(f"Viewing data between {start_time} and {end_time}")
df.loc[start_time:end_time].head()

Viewing data between 2023-01-16 00:04:58.812000 and 2023-01-16 04:04:58.812000


,tool,chamber,val1,val2,val3,val4,val5
ts,,,,,,,
2023-01-16 00:04:58.812,11,5,NaN,NaN,5.206152,3.079318,NaN
2023-01-16 00:15:29.030,25,3,NaN,NaN,5.309610,3.192236,NaN
2023-01-16 00:20:15.537,25,3,NaN,NaN,5.300455,3.179418,NaN
2023-01-16 00:33:15.016,4,3,NaN,NaN,5.180212,3.207801,NaN
2023-01-16 00:34:11.201,16,6,NaN,NaN,5.142064,3.958251,NaN


## 2. Rolling Windows (Moving Averages)
Smoothing high-frequency data to identify underlying trends by removing short-term noise.

In [ ]:
df['val3_smoothed'] = df['val3'].rolling(window='10').mean()
df[['val3', 'val3_smoothed']].head(15)

,val3,val3_smoothed
ts,,
2023-01-16 00:04:58.812,5.206152,NaN
2023-01-16 00:15:29.030,5.309610,NaN
2023-01-16 00:20:15.537,5.300455,NaN
2023-01-16 00:33:15.016,5.180212,NaN
2023-01-16 00:34:11.201,5.142064,NaN
2023-01-16 00:35:54.875,5.130161,NaN
2023-01-16 00:40:31.894,5.158238,NaN
2023-01-16 00:43:44.503,5.164648,NaN
2023-01-16 00:48:25.946,5.222938,NaN


## 3. Extracting Time Components
Breaking down timestamps into hours or days to look for periodicity (e.g., shifts or maintenance cycles).

In [4]:
df['hour'] = df.index.hour
# Note: In a notebook, we can quickly plot distributions
df.groupby('hour')['val3'].mean().head()

hour
0    5.233190
1    5.227853
2    5.232245
3    5.228275
4    5.228653
Name: val3, dtype: float64

## 4. Advanced Resampling
Aggregating data into hourly buckets and calculating multiple statistics at once.

In [7]:
hourly_stats = df['val3'].resample('h').agg(['min', 'max', 'mean'])
hourly_stats.head()

,min,max,mean
ts,,,
2023-01-16 00:00:00,5.095675,5.453963,5.217379
2023-01-16 01:00:00,4.991607,5.430769,5.247617
2023-01-16 02:00:00,5.055391,5.393231,5.228196
2023-01-16 03:00:00,5.008393,5.392926,5.218940
2023-01-16 04:00:00,4.955901,5.402081,5.221010


## 5. Fuzzy Joining with `merge_asof`
This is a critical FAB technique. Use it to join sensor data with event logs (like recipe changes or maintenance) where timestamps don't match exactly. It finds the "last known event" for every sensor reading.

## 5a. `merge_asof` Demo with Synthetic Sensors
A clear mini-example with two short CSVs: `demo_sensor1.csv` and `demo_sensor2.csv`. It shows how `merge_asof` matches each `sensor2` timestamp to the last prior `sensor1` reading.


In [ ]:
sensor1_demo = pd.read_csv("../data/demo_sensor1.csv", parse_dates=["ts"]).sort_values("ts")
sensor1_demo

,ts,sensor1
0,2026-05-02 09:00:00,10
1,2026-05-02 09:00:08,12
2,2026-05-02 09:00:22,11
3,2026-05-02 09:00:35,13
4,2026-05-02 09:00:58,14
5,2026-05-02 09:01:10,15


In [12]:
sensor2_demo = pd.read_csv("../data/demo_sensor2.csv", parse_dates=["ts"]).sort_values("ts")
sensor2_demo

,ts,sensor2
0,2026-05-02 09:00:05,100
1,2026-05-02 09:00:20,105
2,2026-05-02 09:00:30,102
3,2026-05-02 09:00:45,107
4,2026-05-02 09:01:05,108
5,2026-05-02 09:01:20,110


In [ ]:
asof_joined = pd.merge_asof(
    sensor2_demo,
    sensor1_demo,
    on="ts",
    direction="backward",
)

asof_with_tolerance = pd.merge_asof(
    sensor2_demo,
    sensor1_demo,
    on="ts",
    direction="backward",
    tolerance=pd.Timedelta("10s")
)

sensor1_series = sensor1_demo.set_index('ts')['sensor1']
sensor2_demo['sensor1_asof'] = sensor1_series.asof(sensor2_demo['ts']).values

print("merge_asof(sensor2, sensor1, direction='backward'):")
print(asof_joined)

print("\nmerge_asof with 10-second tolerance:")
print(asof_with_tolerance)

print("\nSeries.asof(sensor2['ts']) result:")
print(sensor2_demo)


merge_asof(sensor2, sensor1, direction='backward'):
                   ts  sensor2  sensor1
0 2026-05-02 09:00:05      100       10
1 2026-05-02 09:00:20      105       12
2 2026-05-02 09:00:30      102       11
3 2026-05-02 09:00:45      107       13
4 2026-05-02 09:01:05      108       14
5 2026-05-02 09:01:20      110       15

merge_asof with 10-second tolerance:
                   ts  sensor2  sensor1
0 2026-05-02 09:00:05      100     10.0
1 2026-05-02 09:00:20      105      NaN
2 2026-05-02 09:00:30      102     11.0
3 2026-05-02 09:00:45      107     13.0
4 2026-05-02 09:01:05      108     14.0
5 2026-05-02 09:01:20      110     15.0

Series.asof(sensor2['ts']) result:
                   ts  sensor2  sensor1_asof
0 2026-05-02 09:00:05      100            10
1 2026-05-02 09:00:20      105            12
2 2026-05-02 09:00:30      102            11
3 2026-05-02 09:00:45      107            13
4 2026-05-02 09:01:05      108            14
5 2026-05-02 09:01:20      110            15

## 6. Time Shifting (Delta Analysis)
Calculating the change in sensor value between consecutive readings using `.diff()`.

In [9]:
df['val3_delta'] = df['val3'].diff()
df[['val3', 'val3_delta']].head()

,val3,val3_delta
ts,,
2023-01-16 00:04:58.812,5.206152,NaN
2023-01-16 00:15:29.030,5.309610,0.103458
2023-01-16 00:20:15.537,5.300455,-0.009155
2023-01-16 00:33:15.016,5.180212,-0.120243
2023-01-16 00:34:11.201,5.142064,-0.038148
